In [17]:
!nvidia-smi

Tue Jun  2 19:06:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
from google.colab import drive
drive.mount('/content/drive')

# Create project folders on Drive
import os
os.makedirs('/content/drive/MyDrive/pytorch-code-assistant/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/pytorch-code-assistant/merged', exist_ok=True)
print("Drive mounted and folders created!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and folders created!


In [19]:
!git clone https://github.com/fatimaessaady/pytorch-code-assistant.git
%cd pytorch-code-assistant
!ls

Cloning into 'pytorch-code-assistant'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 62 (delta 34), reused 45 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 18.93 KiB | 2.70 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/pytorch-code-assistant/pytorch-code-assistant
notebooks  README.md  requirements.txt	scripts


In [20]:
!pip install transformers==4.40.0 datasets==2.19.0 accelerate==0.29.3 \
             peft==0.10.0 trl==0.8.6 bitsandbytes==0.43.1 \
             huggingface_hub sentencepiece wandb -q

In [21]:
# Clean uninstall everything conflicting
!pip uninstall bitsandbytes peft trl transformers accelerate -y

# Install all together with matching versions
!pip install -q bitsandbytes==0.45.0
!pip install -q transformers==4.40.0
!pip install -q peft==0.10.0
!pip install -q trl==0.8.6
!pip install -q accelerate==0.29.3
!pip install -q datasets==2.19.0
!pip install -q sentencepiece huggingface_hub wandb

Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
Found existing installation: transformers 4.40.0
Uninstalling transformers-4.40.0:
  Successfully uninstalled transformers-4.40.0
Found existing installation: accelerate 0.29.3
Uninstalling accelerate-0.29.3:
  Successfully uninstalled accelerate-0.29.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [22]:
!apt-get install -q cmake
!pip uninstall bitsandbytes -y
!pip install -q git+https://github.com/TimDettmers/bitsandbytes.git

Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Found existing installation: bitsandbytes 0.45.0
Uninstalling bitsandbytes-0.45.0:
  Successfully uninstalled bitsandbytes-0.45.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [23]:
import torch
import bitsandbytes as bnb
import peft
import trl

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"bitsandbytes: {bnb.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")
print("All good!")

torch: 2.11.0+cu128
CUDA available: True
bitsandbytes: 0.50.0.dev0
peft: 0.10.0
trl: 0.8.6
All good!


In [32]:
# Install huggingface_hub for dataset login
!pip install -q huggingface_hub

# Login first so we can access The Stack
from huggingface_hub import login
login()

In [33]:
# Create the data folders and run the dataset builder
!mkdir -p data/processed data/raw
!python scripts/dataset_builder.py

Building PyTorch instruction dataset

Adding 10 handcrafted examples...
Loading PyTorch samples from The Stack...
Filtering PyTorch files: 10000it [00:06, 1564.33it/s]
Collected 234 samples from The Stack.

Total unique examples: 241
Saved 241 examples → /content/pytorch-code-assistant/pytorch-code-assistant/data/processed/pytorch_dataset.jsonl

Dataset build complete!
Output: /content/pytorch-code-assistant/pytorch-code-assistant/data/processed/pytorch_dataset.jsonl


In [34]:
import os
path = '/content/pytorch-code-assistant/data/processed/pytorch_dataset.jsonl'
print(f"File exists: {os.path.exists(path)}")

# Count examples
with open(path) as f:
    lines = f.readlines()
print(f"Examples: {len(lines)}")

File exists: True
Examples: 241


In [35]:
!git pull

Already up to date.


In [36]:
!python scripts/train.py

bitsandbytes library load error: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 366, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 332, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
GPU: Tesla T4
VRAM: 15.6 GB
Train: 216 | Val: 25
Loading model in float16 (no quantization)...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new d

In [37]:
!python scripts/evaluate.py

bitsandbytes library load error: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 366, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 332, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
Loading base model...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading LoRA adapter

In [38]:
!pip install gradio -q


In [39]:
!python scripts/demo.py

bitsandbytes library load error: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 366, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 332, in get_native_library
    raise RuntimeError(f"Configured {BNB_BACKEND} binary not found at {cuda_binary_path}")
RuntimeError: Configured CUDA binary not found at /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cuda128.so
Loading model...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Model ready!
* Running on